In [1]:
import psycopg2
from kafka import KafkaProducer
import json
import time

In [2]:
producer = KafkaProducer(
    bootstrap_servers="kafka:29092",
    value_serializer=lambda v: json.dumps(v).encode("utf-8")
)

In [3]:
conn = psycopg2.connect(
    host="postgres",
    port=5432,
    database="batch_db",
    user="airflow",
    password="airflow"
)
cur = conn.cursor()

last_id = 0

In [ ]:
while True:
    cur.execute("""
        SELECT id, symbol, date, close, return, volatility_20d
        FROM yahoo_finance.stock_price
        WHERE id > %s
        ORDER BY id
    """, (last_id,))
    
    rows = cur.fetchall()

    for r in rows:
        last_id = r[0]

        msg = {
            "symbol": r[1],
            "date": str(r[2]),
            "price": r[3],
            "return": r[4],
            "volatility": r[5]
        }

        producer.send("yahoo_finance", msg)
        print("📤 Sent:", msg)

    producer.flush()
    time.sleep(5)

📤 Sent: {'symbol': 'AAPL', 'date': '2023-02-01', 'price': 143.00291442871094, 'return': 0.007900626948189604, 'volatility': 0.012757848190815222}
📤 Sent: {'symbol': 'AAPL', 'date': '2023-02-02', 'price': 148.30296325683594, 'return': 0.03706252316114256, 'volatility': 0.014354376878637845}
📤 Sent: {'symbol': 'AAPL', 'date': '2023-02-03', 'price': 151.9215545654297, 'return': 0.02439999329161724, 'volatility': 0.013969144663739335}
📤 Sent: {'symbol': 'AAPL', 'date': '2023-02-06', 'price': 149.1977996826172, 'return': -0.017928692808625968, 'volatility': 0.013955350040413789}
📤 Sent: {'symbol': 'AAPL', 'date': '2023-02-07', 'price': 152.06906127929688, 'return': 0.019244664484245888, 'volatility': 0.01414215501899767}
📤 Sent: {'symbol': 'AAPL', 'date': '2023-02-08', 'price': 149.38465881347656, 'return': -0.017652522105663704, 'volatility': 0.015311635262341854}
📤 Sent: {'symbol': 'AAPL', 'date': '2023-02-09', 'price': 148.35214233398438, 'return': -0.0069117972869048305, 'volatility': 0